# Minimalbeispiel für den Aufruf eines KI-Modells zur Analyse der Bundestagsreden

In [1]:
import jsonlines
import matplotlib.pyplot as plt

import ast


import os
from collections import Counter
from itertools import chain

## Aufruflinks und Funktion für die KIToolbox

## Laden der Reden

In [2]:
from openai import OpenAI

llm_base_url = "https://ki-toolbox.scc.kit.edu/api/v1"
llm_model = "kit.gpt-oss-120b"

try:
    llm_api_key = os.environ["KIT_AI_API_KEY"]
except KeyError as exc:
    raise RuntimeError("Umgebungs‑Variable KIT_AI_API_KEY fehlt") from exc

def call_llm(messages, temperature: float = 0.0) -> str:
    client = OpenAI(api_key=llm_api_key, base_url=llm_base_url)
    try:
        # Send chat request
        resp = client.chat.completions.create(
            model=llm_model,
            messages=messages,
            temperature=temperature)
    except KeyError as exc:
        raise RuntimeError("Error") from exc

    return resp.choices[0].message.content.strip()



In [5]:

legislatur = 20

alleReden = []
with jsonlines.open(f'../../data/speeches_{legislatur}.jsonl') as f:
    for line in f.iter():
        #for line in list(f):
        alleReden.append(line)

alleReden.sort(key = lambda x:x['date'])

In [6]:
rede = alleReden[10000]

system_msg = f"""
Du bist ein Spezialist in politischer Theorie.
"""

user_msg = f"""
Ordne eine Rede aus dem deutschen Bundestag thematisch ein.

Rede:

{rede['text']}
"""

messages=[
        {
            "role": "system",
            "content": system_msg
        },
        {
            "role": "user",
            "content": user_msg
        }
    ]

response = call_llm(messages,temperature=0.9)
print(response)

**Thematische Einordnung der Rede**

| Ebene | Inhalt | Themen‑Kategorie |
|------|--------|-------------------|
| **1. Grundthema** | Kritik am aktuellen Bundeswahlgesetz und Forderung nach einer grundlegenden Reform des Wahlsystems. | **Wahlrecht / Wahlreform** |
| **2. Schwerpunkt** | Überhangmandate – insbesondere die Sonderregelung zugunsten der CSU (keine Ausgleichsmandate) und deren Verzerrung der Mehrheitsverhältnisse im Bundestag. | **Überhang‑/Ausgleichsmandate** |
| **3. Argumentationslinie** | - „Jede Stimme soll gleich viel zählen“ (Prinzip der Stimmgleichheit).<br>‑ Kritik an parteipolitischen Sonderinteressen (CSU, AfD, CDU/CSU‑Koalition).<br>‑ Verweis auf frühere Debatten (Britta Haßelmann, Dobrindt, Pukelsheim). | **Demokratische Grundprinzipien & Gleichheit** |
| **4. Kontext** | Plädoyer im Bundestag, das an frühere Reden (z. B. zur ersten Lesung) anknüpft und das aktuelle Diskussionsthema – die **„gordische Knoten“-Lösung** – adressiert. | **Parlamentarische Debatte